# MLPerf Inference — Whisper-large-v3 / LibriSpeech (Speech, local WSL)

whisper-large-v3 WER on LibriSpeech dev-clean via `openai-whisper` on your RTX 5070 Ti.
Expected **WER ≈ 3–4%**.

> **How to run (in the `mlperf` WSL distro):**
> ```bash
> wsl -d mlperf
> source /root/mlperf/venv/bin/activate
> pip install -q jupyterlab ipykernel
> python -m ipykernel install --user --name mlperf --display-name 'mlperf (venv)'
> jupyter lab --no-browser --ip 0.0.0.0 --port 8888   # open the printed URL
> ```
> Then open this notebook, pick the **mlperf (venv)** kernel, and Run All.
> Assets are cached under `/root/mlperf/…`, so re-runs skip downloads. Works on a fresh
> distro too (it clones + downloads on first run). Needs torch 2.11+cu128 already in the venv.

## 1 · GPU check

In [1]:
import torch; print('cuda',torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

cuda True NVIDIA GeForce RTX 5070 Ti Laptop GPU


## 2 · Deps (ffmpeg via apt) 

In [2]:
%pip install -q openai-whisper jiwer soundfile
import whisper, jiwer; print('whisper + jiwer ready')

Note: you may need to restart the kernel to use updated packages.
whisper + jiwer ready


In [3]:
%%bash
command -v ffmpeg >/dev/null || { apt-get update -qq && apt-get install -y -qq ffmpeg; }
ffmpeg -version | head -1

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers


## 3 · LibriSpeech dev-clean (free, OpenSLR)

In [4]:
%%bash
mkdir -p /root/mlperf/speech && cd /root/mlperf/speech
if [ ! -d LibriSpeech/dev-clean ]; then wget -q -O d.tgz https://www.openslr.org/resources/12/dev-clean.tar.gz && tar xzf d.tgz && rm d.tgz; fi
echo 'utterances:' $(find LibriSpeech/dev-clean -name '*.flac' | wc -l)

utterances: 2703


## 4 · WER eval (N=100)

In [5]:
%%writefile /root/mlperf/speech/whisper_eval.py
import os, glob, time, torch
try:
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)
except Exception as e: print('SDP guard skipped:', e)
import whisper, jiwer
from whisper.normalizers import EnglishTextNormalizer
N=int(os.environ.get('N_UTT','100')); ROOT='/root/mlperf/speech/LibriSpeech/dev-clean'
refs={}
for t in glob.glob(os.path.join(ROOT,'*','*','*.trans.txt')):
    for line in open(t):
        u,x=line.strip().split(' ',1); refs[u]=x
items=[]
for f in sorted(glob.glob(os.path.join(ROOT,'*','*','*.flac'))):
    u=os.path.splitext(os.path.basename(f))[0]
    if u in refs: items.append((f,refs[u]))
items=items[:N]; print('utterances:',len(items))
model=whisper.load_model('large-v3', device='cuda' if torch.cuda.is_available() else 'cpu')
norm=EnglishTextNormalizer(); hyps=[]; gts=[]; asec=0.0; t0=time.time()
for f,ref in items:
    a=whisper.load_audio(f); asec+=len(a)/16000.0
    r=model.transcribe(a, language='en', fp16=torch.cuda.is_available(), verbose=False)
    hyps.append(norm(r['text'])); gts.append(norm(ref))
el=time.time()-t0; wer=jiwer.wer(gts,hyps)
print('================================================')
print(f'WER: {wer*100:.2f}%  ({len(items)} utts, LibriSpeech dev-clean)')
print(f'audio {asec:.1f}s | wall {el:.1f}s | RTF {el/asec:.3f}')
print('================================================')

Overwriting /root/mlperf/speech/whisper_eval.py


In [6]:
%%bash
source /root/mlperf/venv/bin/activate
cd /root/mlperf/speech && N_UTT=100 python whisper_eval.py

utterances: 100


100%|██████████| 238/238 [00:00<00:00, 406.45frames/s]


WER: 5.02%  (100 utts, LibriSpeech dev-clean)
audio 652.5s | wall 101.5s | RTF 0.156


## Done ✅ — whisper-large-v3 WER on your GPU.